# Attention Mechanism — Pure Python
# No NumPy, no PyTorch. Just lists, loops, and the math module.
# 
# This file implements the EXACT same math as 01_attention_mechanism.md
# but every operation is written out as an explicit Python loop so you
# can see what matrix multiplication actually does step by step.

import math
import random

# ─────────────────────────────────────────────────────────────
# SECTION 0: Basic Matrix Utilities
# 
# In PyTorch, you write `A @ B` (matrix multiplication).
# Here we write that operation as nested for-loops so you see
# exactly what is happening: summing products of rows × columns.
# ─────────────────────────────────────────────────────────────

def mat_mul(A, B):
    """
    Multiply two 2D matrices A (m×k) and B (k×n) → result (m×n).

    For each output position (i, j):
      result[i][j] = sum( A[i][k] * B[k][j]  for k in range(inner_dim) )

    This is literally the definition of matrix multiplication.
    In PyTorch: torch.matmul(A, B)  or  A @ B
    """
    m = len(A)        # Rows in A
    k = len(A[0])     # Columns in A  (must equal rows in B)
    n = len(B[0])     # Columns in B

    # Start with a zero matrix of shape (m, n)
    result = [[0.0] * n for _ in range(m)]

    for i in range(m):
        for j in range(n):
            # Dot product of row i of A with column j of B
            result[i][j] = sum(A[i][col] * B[col][j] for col in range(k))

    return result


def transpose(M):
    """
    Flip a matrix so rows become columns and columns become rows.
    A matrix of shape (m, n) becomes (n, m).

    In PyTorch: M.T  or  M.transpose(-2, -1)

    Example:
      [[1, 2, 3],      [[1, 4],
       [4, 5, 6]]  →   [2, 5],
                        [3, 6]]
    """
    rows = len(M)
    cols = len(M[0])
    return [[M[r][c] for r in range(rows)] for c in range(cols)]


def dot(a, b):
    """Dot product of two 1D vectors. sum(a[i]*b[i] for i)"""
    return sum(x * y for x, y in zip(a, b))

def softmax(scores):
    """
    Converts a list of raw scores into a probability distribution (sums to 1).

    We subtract the max first for numerical stability:
    softmax(x) == softmax(x - max(x)) mathematically, but the latter
    avoids overflow from e^(very_large_number).

    In PyTorch: F.softmax(scores, dim=-1)
    """
    max_score = max(scores)
    exps = [math.exp(s - max_score) for s in scores]
    total = sum(exps)
    return [e / total for e in exps]


def make_zeros(rows, cols):
    """Create a (rows × cols) matrix filled with 0.0"""
    return [[0.0] * cols for _ in range(rows)]

def make_random_matrix(rows, cols, scale=0.1):
    """
    Create a matrix filled with small random values.
    In PyTorch: torch.randn(rows, cols) * scale
    Small initial weights prevent exploding activations at the start of training.
    """
    return [[random.gauss(0, scale) for _ in range(cols)] for _ in range(rows)]

def add_vectors(a, b):
    """Element-wise add two vectors: [a0+b0, a1+b1, ...]"""
    return [x + y for x, y in zip(a, b)]

def scale_vector(v, scalar):
    """Multiply every element of a vector by a scalar."""
    return [x * scalar for x in v]

# ─────────────────────────────────────────────────────────────
# Let's verify our matrix utilities work before going further
# ─────────────────────────────────────────────────────────────

A = [[1, 2], [3, 4]]
B = [[5, 6], [7, 8]]

result = mat_mul(A, B)
print("Matrix multiplication:")
print(f"  [[1,2],[3,4]] @ [[5,6],[7,8]] = {result}")
# Expected: [[1*5+2*7, 1*6+2*8], [3*5+4*7, 3*6+4*8]]
#         = [[19, 22], [43, 50]]

print("\nTranspose:")
M = [[1, 2, 3], [4, 5, 6]]
print(f"  {M}")
print(f"  → {transpose(M)}")

print("\nSoftmax:")
scores = [2.0, 1.0, 0.1]
probs = softmax(scores)
print(f"  softmax({scores}) = {[round(p, 4) for p in probs]}")
print(f"  sum = {sum(probs):.6f}")  # Must be exactly 1.0

# ─────────────────────────────────────────────────────────────
# SECTION 1: Scaled Dot-Product Attention
#
# Given three matrices Q (queries), K (keys), V (values),
# compute:
#
#   Attention(Q, K, V) = softmax( Q·Kᵀ / √d_k ) · V
#
# Think of it as:
#   1. Q·Kᵀ  → how similar is each query to each key?
#   2. / √d_k → scale down to prevent softmax from peaking too hard
#   3. softmax → turn similarities into probabilities (attention weights)
#   4. · V   → use those weights to blend the values
# ─────────────────────────────────────────────────────────────

def attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention — pure Python.

    Q: list of shape [seq_len_q, d_k]   — "What am I looking for?"
    K: list of shape [seq_len_k, d_k]   — "What do I contain?"
    V: list of shape [seq_len_k, d_v]   — "What do I give you?"
    mask: list of shape [seq_len_q, seq_len_k] — 1=allow, 0=block (None = no mask)

    Returns:
        output:  [seq_len_q, d_v] — the attended-over values
        weights: [seq_len_q, seq_len_k] — the attention weight matrix
    """
    seq_len_q = len(Q)
    seq_len_k = len(K)
    d_k       = len(Q[0])
    d_v       = len(V[0])

    # ── Step 1: Compute raw similarity scores ──────────────────
    # scores[i][j] = dot(Q[i], K[j])  → how much does query i match key j?
    # We need K transposed so we can do Q @ K^T
    K_T = transpose(K)  # [d_k, seq_len_k]

    # scores = Q @ K^T → shape [seq_len_q, seq_len_k]
    scores = mat_mul(Q, K_T)

    print(f"\nRaw scores (Q·Kᵀ)  shape [{seq_len_q}×{seq_len_k}]:")
    for i, row in enumerate(scores):
        print(f"  Query {i}: {[round(s, 3) for s in row]}")

    # ── Step 2: Scale by √d_k ──────────────────────────────────
    # Without this, scores grow large as d_k grows → softmax collapses to one-hot
    scale = math.sqrt(d_k)
    scores = [[s / scale for s in row] for row in scores]

    # ── Step 3: Apply causal mask (optional) ───────────────────
    # Positions where mask==0 get set to -∞ so softmax makes them ~0
    NEG_INF = float('-inf')
    if mask is not None:
        for i in range(seq_len_q):
            for j in range(seq_len_k):
                if mask[i][j] == 0:
                    scores[i][j] = NEG_INF

    # ── Step 4: Softmax over the key dimension ──────────────────
    # Each row becomes a probability distribution summing to 1
    weights = [softmax(row) for row in scores]

    print(f"\nAttention weights after softmax (each row sums to 1.0):")
    for i, row in enumerate(weights):
        print(f"  Query {i}: {[round(w, 4) for w in row]}")
        print(f"            (sum = {sum(row):.6f})")

    # ── Step 5: Weighted sum of Values ─────────────────────────
    # output[i] = sum( weights[i][j] * V[j]  for j in range(seq_len_k) )
    # For each query, blend all the value vectors weighted by attention
    output = make_zeros(seq_len_q, d_v)
    for i in range(seq_len_q):           # For each query position
        for j in range(seq_len_k):       # For each key/value position
            for d in range(d_v):         # For each value dimension
                output[i][d] += weights[i][j] * V[j][d]

    return output, weights


# ─────────────────────────────────────────────────────────────
# Demo: 3 tokens, 4-dimensional embeddings
# ─────────────────────────────────────────────────────────────

random.seed(42)
seq_len = 3
d_k = 4
d_v = 4

# In a real Transformer these come from linear projections of the input.
# Here we use small random values to keep numbers readable.
Q = make_random_matrix(seq_len, d_k, scale=1.0)
K = make_random_matrix(seq_len, d_k, scale=1.0)
V = make_random_matrix(seq_len, d_v, scale=1.0)

print("=" * 55)
print("DEMO: 3-token sequence, d_k=4, no mask (full attention)")
print("=" * 55)
output, weights = attention(Q, K, V)

print(f"\nOutput (attended values)  shape [{seq_len}×{d_v}]:")
for i, row in enumerate(output):
    print(f"  Token {i}: {[round(x, 4) for x in row]}")

# ─────────────────────────────────────────────────────────────
# SECTION 2: Causal (Masked) Attention
#
# In a GPT-style model, token at position i is NOT allowed to
# look at tokens at positions i+1, i+2, ...
# We enforce this by masking the upper triangle of the scores.
#
# Mask for 4 tokens:
#   [[1, 0, 0, 0],
#    [1, 1, 0, 0],
#    [1, 1, 1, 0],
#    [1, 1, 1, 1]]
#
# Row 0 (first token): can only see itself → weights [1, 0, 0, 0]
# Row 3 (last token):  can see everything  → weights distributed across all 4
# ─────────────────────────────────────────────────────────────

def make_causal_mask(seq_len):
    """
    Creates a lower-triangular matrix of 1s.
    Position [i][j] = 1 means query i CAN attend to key j.
    Position [i][j] = 0 means query i CANNOT attend to key j (future token).
    """
    return [[1 if j <= i else 0 for j in range(seq_len)] for i in range(seq_len)]

seq_len = 4
causal_mask = make_causal_mask(seq_len)

print("\n\n" + "=" * 55)
print("DEMO: Causal (Masked) Attention — 4 tokens")
print("=" * 55)
print("\nCausal mask (1=attend, 0=blocked):")
for i, row in enumerate(causal_mask):
    print(f"  Token {i}: {row}")

Q2 = make_random_matrix(seq_len, d_k, scale=1.0)
K2 = make_random_matrix(seq_len, d_k, scale=1.0)
V2 = make_random_matrix(seq_len, d_v, scale=1.0)

_, masked_weights = attention(Q2, K2, V2, mask=causal_mask)

print("\nVerification — upper triangle must be exactly 0.0:")
for i, row in enumerate(masked_weights):
    for j, w in enumerate(row):
        if j > i and abs(w) > 1e-9:
            print(f"  ERROR: masked_weights[{i}][{j}] = {w} (should be 0!)")
print("  All upper-triangle weights are 0.0 ✓")

# ─────────────────────────────────────────────────────────────
# SECTION 3: Why the √d_k Scaling Matters
#
# As d_k (the embedding dimension) grows, dot products grow larger
# because we're summing more terms. This pushes scores into extreme
# values before softmax, which makes the softmax output near one-hot
# (one weight close to 1, all others close to 0).
#
# A near one-hot attention distribution means:
#   - The gradient through softmax vanishes (saturated region)
#   - The model always focuses on the same token — can't learn subtlety
# ─────────────────────────────────────────────────────────────

def max_attention_weight(d_k, n_tokens=10, trials=100):
    """
    Simulate many random attention computations and measure how peaked
    the output is. Returns the average of the maximum weight per query.
    A value close to 1.0 = collapsed/peaked. Closer to 1/n_tokens = uniform.
    """
    peak_weights = []
    for _ in range(trials):
        Q_trial = make_random_matrix(1, d_k, scale=1.0)
        K_trial = make_random_matrix(n_tokens, d_k, scale=1.0)
        V_trial = make_random_matrix(n_tokens, d_k, scale=1.0)
        _, w = attention(Q_trial, K_trial, V_trial, mask=None)
        peak_weights.append(max(w[0]))
    return sum(peak_weights) / len(peak_weights)

print("\n\n" + "=" * 55)
print("DEMO: Effect of d_k scaling on attention peakiness")
print("=" * 55)
print("(Higher max weight = more peaked = worse gradient flow)")
print(f"{'d_k':>6} | {'Avg max weight (NO scale)':>26} | {'Avg max weight (WITH scale)':>28}")
print("-" * 65)

for d_k_test in [4, 16, 64, 256]:
    # To compare without scale: temporarily override the scale in attention.
    # Instead, we manually compute it here to keep the demo clean.
    random.seed(0)
    Q_t = make_random_matrix(1, d_k_test, scale=1.0)
    K_t = make_random_matrix(8, d_k_test, scale=1.0)
    V_t = make_random_matrix(8, d_k_test, scale=1.0)

    # Without scale: use d_k_test=1 (effectively no division)
    K_T = transpose(K_t)
    raw_scores = mat_mul(Q_t, K_T)[0]

    # No scaling
    no_scale_w = softmax(raw_scores)
    # With scaling
    scaled_scores = [s / math.sqrt(d_k_test) for s in raw_scores]
    scale_w = softmax(scaled_scores)

    print(f"{d_k_test:>6} | {max(no_scale_w):>26.4f} | {max(scale_w):>28.4f}")

print("\n→ Without scaling: large d_k pushes max weight → 1.0 (collapsed)")
print("→ With scaling:    max weight stays moderate regardless of d_k")